# Stage 03 · Step 2 — Held-out test evaluation

Score the trained PPO policy on **test printers (85–99)** under the **same** `scalar_objective` used by Stages 01 & 02. Re-evaluate Stages 01 and 02 on the same printer set so the comparison is apples-to-apples.

Outputs:
- `results/per_printer_tau_test.csv` — Stage 03's per-printer τ + KPIs.
- `results/per_printer_tau_test_stage01.csv`, `..._stage02.csv` — same shape, with the constant τ replicated.
- `results/kpi_comparison.csv` — fleet-level KPI table for all three stages.
- `results/best_tau_per_printer.yaml` — the per-printer τ vectors as YAML.

In [1]:
from __future__ import annotations
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from ml import PROJECT_ROOT
from ml.lib.data import TEST_PRINTERS
from ml.lib.env_runner import default_dates
from ml.lib.objective import INFEASIBLE_FLOOR
from ml.lib.rl import (
    evaluate_constant_tau,
    evaluate_per_printer_policy,
    kpi_comparison_table,
    load_ppo,
    load_ssl_encoder,
    per_printer_table_for_constant_tau,
)
from backend.simulator.generate import load_configs
from backend.simulator.schema import COMPONENT_IDS

STAGE_DIR = PROJECT_ROOT / 'ml_models/03_rl+ssl'
MODELS_DIR = STAGE_DIR / 'models'
RESULTS_DIR = STAGE_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

STAGE_01_BEST = PROJECT_ROOT / 'ml_models/01_baseline/results/best_tau.yaml'
STAGE_02_BEST = PROJECT_ROOT / 'ml_models/02_ssl/results/best_tau_surrogate.yaml'

DATES = default_dates()
components_cfg, couplings_cfg, cities_cfg = load_configs()

## Stage 03: per-printer τ on test printers

In [2]:
model = load_ppo(MODELS_DIR / 'ppo_policy_best.zip')
encoder = load_ssl_encoder()

stage03_per_printer, stage03_kpis = evaluate_per_printer_policy(
    model,
    printer_ids=list(TEST_PRINTERS),
    encoder_bundle=encoder,
    dates=DATES,
    components_cfg=components_cfg,
    couplings_cfg=couplings_cfg,
    cities_cfg=cities_cfg,
)
stage03_per_printer.to_csv(RESULTS_DIR / 'per_printer_tau_test.csv', index=False)

print('Stage 03 fleet KPIs:')
for k, v in stage03_kpis.items():
    print(f'  {k}: {v}')
stage03_per_printer.round(2)

/home/sterry/Desktop/projects/hackupc2026/.venv/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Stage 03 fleet KPIs:
  value: 9991901654.074276
  annual_cost: 3644173.139885026
  availability: 0.050809834592572344
  preventive_cost: 418.609362168081
  corrective_cost: 3643754.5305228583
  deficit: 0.8991901654074276
  horizon_days: 3653
  n_printers: 15


,printer_id,tau_C1,tau_C2,tau_C3,tau_C4,tau_C5,tau_C6,annual_cost,availability,deficit,value
0,85,100.17,500.0,195.26,100.0,761.90,1000.0,3944210.07,0.00,0.95,1.050000e+10
1,86,100.17,500.0,195.59,100.0,761.99,1000.0,3653649.84,0.06,0.89,9.862624e+09
2,87,100.18,500.0,195.25,100.0,762.17,1000.0,3899106.24,0.00,0.95,1.050000e+10
3,88,100.17,500.0,195.53,100.0,762.12,1000.0,4106037.91,0.00,0.95,1.050000e+10
4,89,100.16,500.0,195.18,100.0,761.78,1000.0,3278001.27,0.11,0.84,9.367597e+09
5,90,100.00,500.0,190.83,100.0,759.86,1000.0,3597457.53,0.07,0.88,9.833881e+09
6,91,99.86,500.0,190.20,100.0,759.21,1000.0,3509519.57,0.07,0.88,9.804225e+09
7,92,99.75,500.0,190.17,100.0,759.07,1000.0,3353580.92,0.15,0.80,8.983324e+09
8,93,99.75,500.0,190.29,100.0,759.28,1000.0,3543054.98,0.09,0.86,9.622069e+09
9,94,99.99,500.0,191.63,100.0,760.48,1000.0,3028535.42,0.19,0.76,8.636235e+09


## Stages 01 & 02: constant τ replicated across test printers

In [3]:
stage01_per_printer = None
stage01_kpis = None
if STAGE_01_BEST.exists():
    with STAGE_01_BEST.open() as handle:
        stage01_payload = yaml.safe_load(handle)
    stage01_tau = {c: float(stage01_payload['tau_nom_h'][c]) for c in COMPONENT_IDS}
    stage01_per_printer = per_printer_table_for_constant_tau(
        stage01_tau,
        printer_ids=list(TEST_PRINTERS),
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    stage01_per_printer.to_csv(RESULTS_DIR / 'per_printer_tau_test_stage01.csv', index=False)
    stage01_kpis = evaluate_constant_tau(
        stage01_tau,
        printer_ids=list(TEST_PRINTERS),
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    print('Stage 01 fleet KPIs:')
    for k, v in stage01_kpis.items():
        print(f'  {k}: {v}')
else:
    print(f'{STAGE_01_BEST} not found; skipping Stage 01 eval')

Stage 01 fleet KPIs:
  value: 9923761311.981018
  annual_cost: 3620777.6758828363
  availability: 0.057623868801898105
  preventive_cost: 0.0
  corrective_cost: 3620777.6758828363
  deficit: 0.8923761311981019
  horizon_days: 3653
  n_printers: 15


In [4]:
stage02_per_printer = None
stage02_kpis = None
if STAGE_02_BEST.exists():
    with STAGE_02_BEST.open() as handle:
        stage02_payload = yaml.safe_load(handle)
    stage02_tau = {c: float(stage02_payload['tau_nom_h'][c]) for c in COMPONENT_IDS}
    stage02_per_printer = per_printer_table_for_constant_tau(
        stage02_tau,
        printer_ids=list(TEST_PRINTERS),
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    stage02_per_printer.to_csv(RESULTS_DIR / 'per_printer_tau_test_stage02.csv', index=False)
    stage02_kpis = evaluate_constant_tau(
        stage02_tau,
        printer_ids=list(TEST_PRINTERS),
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    print('Stage 02 fleet KPIs:')
    for k, v in stage02_kpis.items():
        print(f'  {k}: {v}')
else:
    print(f'{STAGE_02_BEST} not found; skipping Stage 02 eval')

Stage 02 fleet KPIs:
  value: 10500000000.0
  annual_cost: 4044316.6965507804
  availability: 0.0
  preventive_cost: 21700.69600328497
  corrective_cost: 4022616.000547495
  deficit: 0.95
  horizon_days: 3653
  n_printers: 15


## Comparison table

Same `scalar_objective`, same printer set, same horizon — three policies.

In [5]:
stage_definitions = []
per_printer_dfs = {'stage_03': stage03_per_printer}
fleet_kpis = {'stage_03': stage03_kpis}

if stage01_kpis is not None:
    stage_definitions.append(('stage_01', stage01_tau, 'Optuna over constant τ'))
    per_printer_dfs['stage_01'] = stage01_per_printer
    fleet_kpis['stage_01'] = stage01_kpis
if stage02_kpis is not None:
    stage_definitions.append(('stage_02', stage02_tau, 'SSL+RUL surrogate over constant τ'))
    per_printer_dfs['stage_02'] = stage02_per_printer
    fleet_kpis['stage_02'] = stage02_kpis
stage_definitions.append(('stage_03', None, 'PPO + frozen SSL encoder, per-printer τ'))

comparison = kpi_comparison_table(
    test_printers=list(TEST_PRINTERS),
    stage_definitions=stage_definitions,
    per_printer_dfs=per_printer_dfs,
    fleet_kpis=fleet_kpis,
)
comparison.to_csv(RESULTS_DIR / 'kpi_comparison.csv', index=False)
comparison

,stage,policy_class,description,fleet_value,fleet_annual_cost,fleet_availability,fleet_deficit,feasible_printer_pct,infeasible_floor_breached,n_test_printers
0,stage_01,constant τ,Optuna over constant τ,9.923761e+09,3.620778e+06,0.057624,0.892376,0.0,True,15
1,stage_02,constant τ,SSL+RUL surrogate over constant τ,1.050000e+10,4.044317e+06,0.000000,0.950000,0.0,True,15
2,stage_03,per-printer τ,"PPO + frozen SSL encoder, per-printer τ",9.991902e+09,3.644173e+06,0.050810,0.899190,0.0,True,15


In [6]:
best_tau_per_printer = {
    int(row['printer_id']): {c: float(row[f'tau_{c}']) for c in COMPONENT_IDS}
    for _, row in stage03_per_printer.iterrows()
}
payload = {
    'evaluated_on': 'test printers (id 85..99)',
    'fleet_value': float(stage03_kpis['value']),
    'fleet_annual_cost_eur_per_printer_year': float(stage03_kpis['annual_cost']),
    'fleet_availability': float(stage03_kpis['availability']),
    'fleet_deficit': float(stage03_kpis['deficit']),
    'tau_per_printer': best_tau_per_printer,
}
with (RESULTS_DIR / 'best_tau_per_printer.yaml').open('w', encoding='utf-8') as handle:
    yaml.safe_dump(payload, handle, sort_keys=False)
print('Wrote best_tau_per_printer.yaml')
print(yaml.safe_dump(
    {k: v for k, v in payload.items() if k != 'tau_per_printer'},
    sort_keys=False,
))

Wrote best_tau_per_printer.yaml
evaluated_on: test printers (id 85..99)
fleet_value: 9991901654.074276
fleet_annual_cost_eur_per_printer_year: 3644173.139885026
fleet_availability: 0.050809834592572344
fleet_deficit: 0.8991901654074276



## Done criteria

- [ ] `kpi_comparison.csv` shows `stage_03.fleet_value < stage_02.fleet_value`.
- [ ] `stage_03.fleet_availability >= 0.95` (no infeasibility floor breach).
- [ ] Per-printer τ heatmap (next notebook) shows actual variation across printers — otherwise the policy collapsed to a constant and Stage 03 reduces to Stage 02.